In [9]:
import boto3
import os
from pathlib import Path

In [5]:
s3 = boto3.client('s3')

def upload_file_to_s3(file_name, bucket, object_name=None):
    """Upload a file to an S3 bucket

    :param file_name: File to upload
    :param bucket: Bucket to upload to
    :param object_name: S3 object name. If not specified then file_name is used
    :return: True if file was uploaded, else False
    """
    if object_name is None:
        object_name = file_name

    try:
        s3.upload_file(file_name, bucket, object_name)
    except Exception as e:
        print(f"Error uploading file: {e}")
        return False
    return True


In [6]:


dims = ['../../data/bronze/dim_golf_course.csv', '../../data/bronze/dim_golf_course_holes.csv' ]

bucket_name = "fourball-qualifying-342262164553-us-east-1-an"
sub_bucket = 'dims'
for dim in dims:
    key = f"{sub_bucket}/{os.path.basename(dim)}"
    upload_file_to_s3(dim, bucket_name, key)
    print(f"Uploaded {dim} to s3://{bucket_name}/{key}")




Uploaded ../../data/bronze/dim_golf_course.csv to s3://fourball-qualifying-342262164553-us-east-1-an/dims/dim_golf_course.csv
Uploaded ../../data/bronze/dim_golf_course_holes.csv to s3://fourball-qualifying-342262164553-us-east-1-an/dims/dim_golf_course_holes.csv


In [ ]:
SCORES = Path.cwd().parents[1] / "data/bronze/scores"
existing = {o["Key"] for p in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket_name, Prefix="scores/") for o in p.get("Contents", [])}

for path in SCORES.glob("*-fourball/**/*"):
    key = f"scores/{path.relative_to(SCORES).as_posix()}"
    if path.is_file() and key not in existing:  # only new files, so Snowpipe never double-loads
        upload_file_to_s3(str(path), bucket_name, key)
        print(f"Uploaded {path} to s3://{bucket_name}/{key}")